In [34]:
pip install requests pandas mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import requests
import json
import os
import pandas as pd
import mysql.connector

In [36]:
# -------- API --------
API_HOST = "aerodatabox.p.rapidapi.com"
API_KEY = "62379da2bfmshffc41ab8de94263p18db2fjsn3bf5a0d21607"

HEADERS = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": API_HOST
}


In [37]:

# -------- AIRPORTS --------
airports = ["DEL", "BOM", "BLR", "HYD", "MAA",
            "DXB", "LHR", "JFK", "SIN", "FRA",
            "DOH", "CDG", "AMS", "HKG", "LAX"]

# -------- COMMON FUNCTION --------

def get_data(url, name):
    file = f"data/{name}.json"
    
    if os.path.exists(file):
        with open(file) as f:
            return json.load(f)
    
    response = requests.get(url, headers=HEADERS)
    data = response.json()
    
    os.makedirs("data", exist_ok=True)
    with open(file, "w") as f:
        json.dump(data, f)
    
    return data

# -------- FETCH AIRPORT DATA --------
airport_list = []

for code in airports:
    url = f"https://{API_HOST}/airports/iata/{code}"
    data = get_data(url, f"airport_{code}")
    
    airport_list.append({
        "iata_code": data.get("iata"),
        "name": data.get("fullName"),
        "city": data.get("municipalityName"),
        "country": data.get("country", {}).get("name"),
        "timezone": data.get("timeZone")
    })

# -------- CREATE DATAFRAME --------
df_airport = pd.DataFrame(airport_list)

# -------- SAVE AS CSV (ONLY THIS) --------
df_airport.to_csv("data/airports.csv", index=False)

# -------- PREVIEW --------
print(df_airport.head())

  iata_code                        name       city country      timezone
0       DEL     New Delhi Indira Gandhi  New Delhi   India  Asia/Kolkata
1       BOM  Mumbai Chhatrapati Shivaji     Mumbai   India  Asia/Kolkata
2       BLR         Bangalore Bengaluru  Bangalore   India  Asia/Kolkata
3       HYD      Hyderabad Rajiv Gandhi  Hyderabad   India  Asia/Kolkata
4       MAA                     Chennai    Chennai   India  Asia/Kolkata


In [38]:
#------------- FLIGHT DATA --------------

flights_list = []

for code in airports:
    data = get_data(
        f"https://{API_HOST}/flights/airports/iata/{code}?withLeg=true&direction=Departure&limit=10",
        f"flights_{code}"
    )

    for f in data.get("departures", []):

        departure_time = f.get("departure", {}).get("scheduledTime", {})
        arrival_time = f.get("arrival", {}).get("scheduledTime", {})

        flights_list.append({
            "flight_number": f.get("number"),
            "airline_name": f.get("airline", {}).get("name"),
            "airline_iata": f.get("airline", {}).get("iata"),
            "origin": code,
            "destination": f.get("arrival", {}).get("airport", {}).get("iata"),
            "departure_time_utc": departure_time.get("utc"),
            "arrival_time_utc": arrival_time.get("utc"),
            "status": f.get("status"),
            "aircraft_registration": f.get("aircraft", {}).get("reg")
        })

df_flights = pd.DataFrame(flights_list)

#-------------- Save CSV --------------

df_flights.to_csv("data/flights.csv", index=False)

print("Flights saved to CSV.")
print(df_flights.head())


Flights saved to CSV.
  flight_number airline_name airline_iata origin destination  \
0       SG 5011     SpiceJet           SG    DEL         NaN   
1       AI 2425    Air India           AI    DEL         BOM   
2       6E 2136       IndiGo           6E    DEL         CCU   
3       AI 2859    Air India           AI    DEL         HYD   
4       6E 6019       IndiGo           6E    DEL         HYD   

  departure_time_utc   arrival_time_utc    status aircraft_registration  
0  2026-03-18 05:05Z                NaN  Departed                VT-SXA  
1  2026-03-18 05:05Z  2026-03-18 07:35Z  Departed                VT-TNI  
2  2026-03-18 05:11Z  2026-03-18 07:10Z  Departed                VT-NCN  
3  2026-03-18 05:12Z  2026-03-18 07:15Z  Departed                   NaN  
4  2026-03-18 05:13Z  2026-03-18 07:30Z  Departed                VT-NHO  


In [39]:
df_flights.head(5)
df_flights.tail(5)

,flight_number,airline_name,airline_iata,origin,destination,departure_time_utc,arrival_time_utc,status,aircraft_registration
11774,AS 798,Alaska,AS,LAX,SEA,2026-03-18 17:00Z,2026-03-18 19:58Z,Expected,NaN
11775,AA 2546,American,AA,LAX,MEX,2026-03-18 17:01Z,2026-03-18 20:40Z,Expected,NaN
11776,NK 1641,Spirit,NK,LAX,LAS,2026-03-18 17:03Z,2026-03-18 18:10Z,Expected,NaN
11777,DL 3951,Delta Air Lines,DL,LAX,PHX,2026-03-18 17:05Z,2026-03-18 18:29Z,Expected,NaN
11778,AA 6376,American,AA,LAX,SAF,2026-03-18 17:05Z,2026-03-18 19:12Z,Expected,NaN


In [40]:
def get_data(url, name):
    file = f"data/{name}.json"
    os.makedirs("data", exist_ok=True)

    if os.path.exists(file):
        return json.load(open(file))

    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        return {}

    data = response.json()

    json.dump(data, open(file, "w"))
    return data

In [41]:
# ------------Aircraft ---------------

import time

aircraft_list = []

start_time = time.time()
registrations = df_flights["aircraft_registration"].dropna().unique()

for reg in registrations:

    # ⛔ Stop after 15 minutes
    if time.time() - start_time > 900:
        print("⏰ Time limit reached. Stopping...")
        break

    data = get_data(
        f"https://{API_HOST}/aircrafts/reg/{reg}",
        f"aircraft_{reg}"
    )

    # ✅ Skip empty responses directly
    if not data:
        continue

    aircraft_list.append({
        "registration": data.get("reg"),
        "model": data.get("model")
    })

# Create DataFrame
df_aircraft = pd.DataFrame(aircraft_list).drop_duplicates()

# Save to CSV
df_aircraft.to_csv("data/aircraft.csv", index=False)

print("Aircraft saved to CSV.")

Aircraft saved to CSV.


In [42]:
df_aircraft

,registration,model
0,VT-SXA,B738
1,VT-TNI,A20N
2,VT-NCN,A21N
3,NaN,NaN
4,VT-SCV,A319
...,...,...
896,V8-RBA,A20N
897,B-KJD,B738
898,B-329N,A20N
899,B-LVF,NaN


In [43]:
from datetime import datetime

# Convert to datetime

df_flights["departure_time_utc"] = pd.to_datetime(df_flights["departure_time_utc"])
df_flights["arrival_time_utc"] = pd.to_datetime(df_flights["arrival_time_utc"])

# Calculate duration in minutes

df_flights["delay_minutes"] = (
    df_flights["arrival_time_utc"] - df_flights["departure_time_utc"]
).dt.total_seconds() / 60

In [44]:
df_flights[["departure_time_utc", "arrival_time_utc", "delay_minutes"]]

df_flights.head(5)

,flight_number,airline_name,airline_iata,origin,destination,departure_time_utc,arrival_time_utc,status,aircraft_registration,delay_minutes
0,SG 5011,SpiceJet,SG,DEL,NaN,2026-03-18 05:05:00+00:00,NaT,Departed,VT-SXA,NaN
1,AI 2425,Air India,AI,DEL,BOM,2026-03-18 05:05:00+00:00,2026-03-18 07:35:00+00:00,Departed,VT-TNI,150.0
2,6E 2136,IndiGo,6E,DEL,CCU,2026-03-18 05:11:00+00:00,2026-03-18 07:10:00+00:00,Departed,VT-NCN,119.0
3,AI 2859,Air India,AI,DEL,HYD,2026-03-18 05:12:00+00:00,2026-03-18 07:15:00+00:00,Departed,NaN,123.0
4,6E 6019,IndiGo,6E,DEL,HYD,2026-03-18 05:13:00+00:00,2026-03-18 07:30:00+00:00,Departed,VT-NHO,137.0


Insert the data  (python ----> SQL)

In [45]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
# -------- DB --------

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Deepanshu@1",
    database="flight_db"
)

cursor = conn.cursor()
print("Connected Successfully")

Connected Successfully


In [47]:
from sqlalchemy import create_engine

engine = create_engine(
    "mysql+mysqlconnector://root@localhost:3306/flight_db",
    connect_args={"password": "Deepanshu@1"}
)

In [48]:

df_flights.to_sql("flights", engine, if_exists="append", index=False)
print("flight Data Inserted Successfully")

flight Data Inserted Successfully


In [49]:
# Convert datetime
df_flights["departure_time_utc"] = pd.to_datetime(df_flights["departure_time_utc"], errors="coerce")
df_flights["arrival_time_utc"] = pd.to_datetime(df_flights["arrival_time_utc"], errors="coerce")


df_flights.to_sql("flights", engine, if_exists="append", index=False)
print("flight Data Inserted Successfully")

flight Data Inserted Successfully


In [52]:
df_airport.to_sql("airport", engine, if_exists="append", index=False)
print("Airport Data Inserted Successfully")


Airport Data Inserted Successfully


In [53]:
df_aircraft.to_sql("aircraft", engine, if_exists="append", index=False)
print("aircraft Data Inserted Successfully")

aircraft Data Inserted Successfully
